# MID data check

Work through four questions: pathology without neck dissection, missing
observations, consistency, and the meaning of the recorded consensus.

The calculations are shared in `scripts/data_checks.py`; this notebook selects
the MID workbook and presents its results. **Run all cells** to refresh the report.
All patients remain included. Expand a code cell, evidence table or settings panel
when you need more detail.

An observation is one **patient–side–level** value. The main-level summaries use
**I, II, III, IV, V and VII on both sides**. Unknown values remain unknown.

In [13]:
DATASET_NAME = "MID"
WORKBOOK_PATH = "updated_datasets/MID_update.xlsx"

In [14]:
from pathlib import Path
import sys

import pandas as pd
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "updated_datasets").exists():
    PROJECT_DIR = PROJECT_DIR / "Mexico_data_analysis"

if str(PROJECT_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR.resolve()))

import importlib
from scripts import data_checks as checks
importlib.reload(checks)
from scripts import notebook_display as report
importlib.reload(report)
from scripts.notebook_display import SOURCE_LABELS, comparison_view, show_note, show_table

DATA_FILE = PROJECT_DIR / WORKBOOK_PATH
source_data = pd.read_excel(DATA_FILE, header=[0, 1, 2])
analysis = checks.analyse_dataset(source_data)
raw = analysis["raw"]
without = "max_llh_without_pathology"
with_pathology = "max_llh_with_pathology"
show_note(f"{DATASET_NAME} · {len(raw)} patients", f"Source: {DATA_FILE.name}")

## 1. Pathology without neck dissection

Find patients with at least one recorded pathology result whose neck-dissection
flag is False or missing. Both positive and negative pathology count, including
results recorded only at sublevels.

Read each patient's recorded levels alongside the centre's comments. A biopsy
can explain pathology without a neck dissection; the finding is a request for
context, not an automatic correction of the flag.

In [15]:
pathology_without_neck_dissection = analysis["pathology_without_neck_dissection"]
show_note(
    f"{len(pathology_without_neck_dissection)} of {len(raw)} patients have pathology without dissection marked True",
    "This can reflect a biopsy. Review the centre's comments before changing a flag.",
)

def compact_pathology(text):
    by_side = {}
    for item in text.split("; "):
        side, observation = item.split(" ", 1)
        level, value = observation.split("=", 1)
        by_side.setdefault(side, []).append(level + (" +" if value == "positive" else " −"))
    return "; ".join(f"{side}: {', '.join(values)}" for side, values in by_side.items())

pathology_view = pathology_without_neck_dissection.copy()
pathology_view["neck_dissection"] = pathology_view["neck_dissection"].map(
    {True: "Yes", False: "No"}
).fillna("Unknown")
pathology_view["pathology_observations"] = pathology_view["pathology_observations"].map(compact_pathology)
show_table(pathology_view.rename(columns={
    "patient_id": "Patient", "neck_dissection": "Neck dissection",
    "pathology_observations": "Pathology (+ / −)", "comments": "Centre's comments",
}))

Patient,Neck dissection,Pathology (+ / −),Centre's comments
MID-10,No,"ipsi: I −, Ia −, Ib −, II −, IIa −, IIb −, III −, IV −, V −, VII −; contra: I −, Ia −, Ib −, II −, IIa −, IIb −, III −, IV −, V −, VII −","PACIENTE SOLO RECIBIO 28 DE 35 FRACCIONES, BUENA RESPUESTA, REVAL POR ONCOQX RADICALIZAN SIN INFORMACIÓN COMPLETA DE LA RESECCIÓN, ESTADO DE REMISION"


## 2. Missing observations

For each source, count patients with any data and known versus missing main-level
values. There are **12 possible values per patient**: six levels on each side.
An absent modality block is marked separately from a block with missing entries.

Coverage describes the information available for later comparisons. It does not
imply that every patient should have undergone every test. Missing observations
are never counted as negative results.

In [16]:
missing_observations = analysis["missing_observations"]
missing_view = missing_observations.copy()
missing_view["Patients with data"] = missing_view["patients_with_data"].map(lambda n: f"{n}/{len(raw)}")
missing_view["Missing values"] = missing_view["missing_pairs"].map(
    lambda n: f"{n}/{len(raw) * 12}"
)
missing_view["modality"] = missing_view["modality"].replace({"diagnostic_consensus": "Recorded consensus"})
missing_view.loc[missing_view["columns_present"].eq(0), "modality"] += " (absent)"
show_table(missing_view[["modality", "Patients with data", "Missing values", "coverage_pct"]].rename(
    columns={"modality": "Source", "coverage_pct": "Coverage"}
), percentages=["Coverage"])

Source,Patients with data,Missing values,Coverage
Recorded consensus,41/42,12/504,97.6%
pathology,2/42,491/504,2.6%
CT,41/42,12/504,97.6%
MRI,0/42,504/504,0.0%
PET,0/42,504/504,0.0%
FNA,3/42,501/504,0.6%
pCT (absent),0/42,504/504,0.0%


## 3. Consistency

Check I against Ia/Ib and II against IIa/IIb within each modality. A positive
child requires a positive parent; two negative children require a negative
parent. One negative child with an unknown sibling is insufficient.

Also run the same reduced checks for identity, valid values, N-stage and node
counts used in the original notebook. Findings are grouped below. Invalid binary
entries are reported here and treated as unknown in comparisons and reconstruction.
Consensus–pathology disagreements are examined separately in section 4.

In [17]:
hierarchy_issues = analysis["hierarchy_issues"]
sanity_issues = analysis["sanity_issues"]
consistency_summary = analysis["consistency_summary"]

total_flags = len(hierarchy_issues) + len(sanity_issues)
show_note(
    "No consistency flags" if total_flags == 0 else f"{total_flags} consistency findings",
    f"Main/sublevel conflicts: {len(hierarchy_issues)}. Reduced sanity findings: {len(sanity_issues)}.",
)
if not hierarchy_issues.empty:
    show_table(hierarchy_issues, details="Main/sublevel findings")
if not sanity_issues.empty:
    show_table(sanity_issues[["patient_id", "check", "category", "details"]],
               details="Reduced sanity-check findings")

row,patient_id,modality,side,level,parent_value,sublevels
16,MID-17,diagnostic_consensus,ipsi,II,True,"IIa=False, IIb=False"


patient_id,check,category,details
MID-4,n_positive_without_localization,Información incompleta,"N1 sin afectación positiva registrada en todas las modalidades y recuentos positivos disponibles. Regiones sin información negativa suficiente: ipsilateral I, ipsilateral II, ipsilateral III, ipsilateral IV, ipsilateral V, ipsilateral VII, contralateral I, contralateral II, contralateral III, contralateral IV, contralateral V, contralateral VII. Indicar dónde se documentó la afectación ganglionar. Confirmar si el estadio es clínico o patológico y que los datos correspondan a la misma evaluación."
MID-6,n_positive_all_regions_negative,Revisar contexto,"N1 sin afectación positiva en todas las modalidades y recuentos positivos disponibles, con observaciones negativas para todas las regiones comprobadas en ambos lados. Verificar el estadio, los valores negativos y si la afectación estaba fuera de estas regiones. Confirmar si el estadio es clínico o patológico y que los datos correspondan a la misma evaluación."
MID-10,dissection_flag_vs_counts,Posible contradicción,"Disección cervical = False, pero se registran ganglios disecados. Verificar cirugía y recuentos."
MID-12,n_positive_all_regions_negative,Revisar contexto,"N1 sin afectación positiva en todas las modalidades y recuentos positivos disponibles, con observaciones negativas para todas las regiones comprobadas en ambos lados. Verificar el estadio, los valores negativos y si la afectación estaba fuera de estas regiones. Confirmar si el estadio es clínico o patológico y que los datos correspondan a la misma evaluación."
MID-13,n_positive_all_regions_negative,Revisar contexto,"N1 sin afectación positiva en todas las modalidades y recuentos positivos disponibles, con observaciones negativas para todas las regiones comprobadas en ambos lados. Verificar el estadio, los valores negativos y si la afectación estaba fuera de estas regiones. Confirmar si el estadio es clínico o patológico y que los datos correspondan a la misma evaluación."
MID-16,n_positive_all_regions_negative,Revisar contexto,"N1 sin afectación positiva en todas las modalidades y recuentos positivos disponibles, con observaciones negativas para todas las regiones comprobadas en ambos lados. Verificar el estadio, los valores negativos y si la afectación estaba fuera de estas regiones. Confirmar si el estadio es clínico o patológico y que los datos correspondan a la misma evaluación."
MID-18,n_positive_all_regions_negative,Revisar contexto,"N1 sin afectación positiva en todas las modalidades y recuentos positivos disponibles, con observaciones negativas para todas las regiones comprobadas en ambos lados. Verificar el estadio, los valores negativos y si la afectación estaba fuera de estas regiones. Confirmar si el estadio es clínico o patológico y que los datos correspondan a la misma evaluación."
MID-18,dissection_flag_vs_counts,Posible contradicción,"Disección cervical = False, pero se registran ganglios disecados. Verificar cirugía y recuentos."
MID-22,dissection_flag_vs_counts,Posible contradicción,"Disección cervical = False, pero se registran ganglios disecados. Verificar cirugía y recuentos."
MID-40,n_positive_all_regions_negative,Revisar contexto,"N2 sin afectación positiva en todas las modalidades y recuentos positivos disponibles, con observaciones negativas para todas las regiones comprobadas en ambos lados. Verificar el estadio, los valores negativos y si la afectación estaba fuera de estas regiones. Confirmar si el estadio es clínico o patológico y que los datos correspondan a la misma evaluación."


## 4. What does the recorded consensus represent?

### Recorded consensus versus each source

Compare the recorded consensus with pathology, CT, MRI, PET, FNA and pCT at the
same patient, side and main level. Only pairs with **both values known** enter
the denominator; the patient column counts people with a disagreement.

Agreement percentages include both positive and negative agreement. High overall
agreement can coexist with important positive-case disagreements, so the case
evidence below remains essential. No comparable pairs means no percentage.

In [18]:
recorded_vs_modalities = analysis["recorded_vs_modalities"]
pathology_result = recorded_vs_modalities.iloc[0]
show_note(
    f"Consensus–pathology disagreements: {int(pathology_result['disagreements'])} / "
    f"{int(pathology_result['compared_pairs'])} known pairs",
    f"Patients affected: {int(pathology_result['patients_affected'])}. Individual cases appear below.",
)
show_table(comparison_view(recorded_vs_modalities, ["Pathology", "CT", "MRI", "PET", "FNA", "pCT"]),
           percentages=["Agreement"])

Comparison,Known pairs,Disagreements,Patients affected,Agreement
Pathology,13,2,1,84.6%
CT,492,24,7,95.1%
MRI,0,0,0,—
PET,0,0,0,—
FNA,3,0,0,100.0%
pCT,0,0,0,—


### Reconstruct consensus with and without pathology

**Without pathology:** CT, MRI, PET, pCT and FNA, where available. **With pathology:**
the same inputs plus pathology. Both use `max_llh`; the recorded consensus is
excluded from both, and FNA remains included.


<details>
<summary>Reconstruction method notes</summary>

Both runs use the same installed lydata defaults and I/II subdivisions. The source observations are preserved. Parent or child labels can be inferred; otherwise, no input means an unknown label.

Default pathology sensitivity and specificity are 1.0. Agreement of a reconstruction containing pathology with that same input is expected by construction.

</details>

In [19]:
modality_settings = analysis["modality_settings"]
reconstruction_summary = analysis["reconstruction_summary"]
show_table(comparison_view(reconstruction_summary, [
    "Recorded vs without pathology", "Recorded vs with pathology", "Without vs with pathology",
]), percentages=["Agreement"])

settings_view = modality_settings.rename(columns={
    "modality": "Modality", "available": "Present", "sensitivity": "Sensitivity",
    "specificity": "Specificity",
})
settings_view["Present"] = settings_view["Present"].map({True: "Yes", False: "No"})
show_table(settings_view, details="Model assumptions: sensitivity and specificity")

Comparison,Known pairs,Disagreements,Patients affected,Agreement
Recorded vs without pathology,492,24,7,95.1%
Recorded vs with pathology,492,23,7,95.3%
Without vs with pathology,492,3,1,99.4%


Modality,Present,Sensitivity,Specificity
CT,Yes,0.81,0.76
MRI,Yes,0.81,0.63
PET,Yes,0.79,0.86
pCT,No,0.81,0.86
FNA,Yes,0.8,0.98
pathology,Yes,1,1


### Which reconstruction agrees with the recorded consensus?

First identify known labels that change when pathology is added. Among those
with known pathology and recorded consensus, count how often the recorded
consensus agrees with each reconstruction. This focuses on the observations
that can distinguish a clinical consensus from one incorporating pathology.

The main case table includes disagreements involving pathology or the
reconstructed consensuses. Ordinary differences with an individual modality
appear only in the expandable full evidence table.
The expanded pathology comparison uses the same known pairs for all three
consensuses. Agreement alone cannot confirm the intended definition or timing.

**+** positive · **−** negative · **—** unknown.

The **Hierarchy** column explains parent changes, for example
`I − → + (Ia −, Ib +)`. Expand **Parent and sublevel evidence: before
and after hierarchy** below the main table to inspect the raw sublevels
and each reconstruction before and after reconciliation.


In [20]:
matched_pathology_summary = analysis["matched_pathology_summary"]
reconstruction_changed = analysis["reconstruction_changed"]
interpretation_summary = analysis["interpretation_summary"]
show_note(
    f"Labels changed by pathology: {int(reconstruction_changed.to_numpy().sum())}",
    f"Patients affected: {int(reconstruction_changed.any(axis=1).sum())}.",
)
interpretation_view = pd.DataFrame({
    "Reconstruction": ["Without pathology", "With pathology"],
    "Agrees with recorded consensus on changed pairs": [
        f"{row.compared_pairs - row.disagreements}/{row.compared_pairs}"
        if row.compared_pairs else "—"
        for row in interpretation_summary.itertuples(index=False)
    ],
})
show_table(interpretation_view)

# Evidence rendering is shared across all dataset notebooks.
report.show_disagreement_evidence(analysis)
show_table(comparison_view(matched_pathology_summary, [
    "Recorded consensus vs pathology", "Without pathology vs pathology", "With pathology vs pathology",
]), percentages=["Agreement"], details="Agreement with pathology on the same known pairs")


Reconstruction,Agrees with recorded consensus on changed pairs
Without pathology,1/3
With pathology,2/3


Patient,Location,Hierarchy,Recorded,Pathology,Without path.,With path.,Raw modalities
MID-1,ipsi IV,—,−,<NA>,+,+,CT +
MID-1,ipsi V,—,−,<NA>,+,+,CT +
MID-10,ipsi I,—,−,−,+,−,CT +
MID-10,ipsi II,—,+,−,+,−,"CT +, FNA +"
MID-10,ipsi III,—,−,−,+,−,CT +
MID-10,contra II,—,+,−,−,−,CT −
MID-22,ipsi I,—,+,<NA>,−,−,CT −
MID-22,ipsi IV,—,+,<NA>,−,−,CT −
MID-22,ipsi V,—,+,<NA>,−,−,CT −
MID-24,ipsi I,—,+,<NA>,−,−,CT −


Patient,Side,Level,Recorded consensus,Pathology,CT,MRI,PET,FNA,Without pathology,With pathology,Centre's comments
MID-1,ipsi,IV,−,<NA>,+,<NA>,<NA>,<NA>,+,+,"R1 + ILVS, PERDIO CONCOMITANCIA POR NEGATIVA DE OM (ESPACIOS)"
MID-1,ipsi,V,−,<NA>,+,<NA>,<NA>,<NA>,+,+,"R1 + ILVS, PERDIO CONCOMITANCIA POR NEGATIVA DE OM (ESPACIOS)"
MID-10,ipsi,I,−,−,+,<NA>,<NA>,<NA>,+,−,"PACIENTE SOLO RECIBIO 28 DE 35 FRACCIONES, BUENA RESPUESTA, REVAL POR ONCOQX RADICALIZAN SIN INFORMACIÓN COMPLETA DE LA RESECCIÓN, ESTADO DE REMISION"
MID-10,ipsi,II,+,−,+,<NA>,<NA>,+,+,−,"PACIENTE SOLO RECIBIO 28 DE 35 FRACCIONES, BUENA RESPUESTA, REVAL POR ONCOQX RADICALIZAN SIN INFORMACIÓN COMPLETA DE LA RESECCIÓN, ESTADO DE REMISION"
MID-10,ipsi,III,−,−,+,<NA>,<NA>,<NA>,+,−,"PACIENTE SOLO RECIBIO 28 DE 35 FRACCIONES, BUENA RESPUESTA, REVAL POR ONCOQX RADICALIZAN SIN INFORMACIÓN COMPLETA DE LA RESECCIÓN, ESTADO DE REMISION"
MID-10,contra,II,+,−,−,<NA>,<NA>,<NA>,−,−,"PACIENTE SOLO RECIBIO 28 DE 35 FRACCIONES, BUENA RESPUESTA, REVAL POR ONCOQX RADICALIZAN SIN INFORMACIÓN COMPLETA DE LA RESECCIÓN, ESTADO DE REMISION"
MID-22,ipsi,I,+,<NA>,−,<NA>,<NA>,<NA>,−,−,"FALLECIO EN TTO SISTEMICO PALIATIVO, SIEMPRE 1ARIO, SIN N+ O METS"
MID-22,ipsi,IV,+,<NA>,−,<NA>,<NA>,<NA>,−,−,"FALLECIO EN TTO SISTEMICO PALIATIVO, SIEMPRE 1ARIO, SIN N+ O METS"
MID-22,ipsi,V,+,<NA>,−,<NA>,<NA>,<NA>,−,−,"FALLECIO EN TTO SISTEMICO PALIATIVO, SIEMPRE 1ARIO, SIN N+ O METS"
MID-24,ipsi,I,+,<NA>,−,<NA>,<NA>,<NA>,−,−,SIN SECUELAS REPORTADAS


Patient,Location,Source,Stage,Parent,Sublevels
MID-10,ipsi I,Recorded consensus,Recorded,−,"Ia −, Ib −"
MID-10,ipsi I,Pathology,Recorded,−,"Ia −, Ib −"
MID-10,ipsi I,CT,Recorded,+,"Ia −, Ib +"
MID-10,ipsi I,Without pathology,Before hierarchy,+,"Ia −, Ib +"
MID-10,ipsi I,Without pathology,After hierarchy,+,"Ia −, Ib +"
MID-10,ipsi I,With pathology,Before hierarchy,−,"Ia −, Ib −"
MID-10,ipsi I,With pathology,After hierarchy,−,"Ia −, Ib −"
MID-10,ipsi II,Recorded consensus,Recorded,+,"IIa +, IIb +"
MID-10,ipsi II,Pathology,Recorded,−,"IIa −, IIb −"
MID-10,ipsi II,CT,Recorded,+,"IIa +, IIb +"


Comparison,Known pairs,Disagreements,Patients affected,Agreement
Recorded consensus vs pathology,13,2,1,84.6%
Without pathology vs pathology,13,3,1,76.9%
With pathology vs pathology,13,0,0,100.0%


### N-stage versus involvement

Compare the recorded N-stage with involvement from the recorded consensus,
both reconstructions and pathology. Flag **N0 with a positive level** and
**N+ without an observed positive**. Distinguish complete negative coverage
from partial negative or absent observations.

The expanded overview lists positive levels on each side for every patient.
The N-stage basis remains unconfirmed unless its own field or prefix specifies
it; a T-stage prefix does not establish an N-stage basis. These are N0/N+
review prompts, not a reconstruction of exact TNM substages. Different sources
may also describe different assessments.

In [21]:
n_stage_details = analysis["n_stage_details"]
n_stage_overview = analysis["n_stage_overview"]
n_stage_review = analysis["n_stage_review"]
review_patients = n_stage_review["patient_id"].unique()
show_note(
    f"N-stage review: {len(review_patients)} " + ("patient" if len(review_patients) == 1 else "patients"),
    ", ".join(map(str, review_patients)) if len(review_patients) else "No N0/N+ review findings.",
)
if not n_stage_review.empty:
    stage_review_view = n_stage_review[[
        "patient_id", "n_stage", "source", "known_pairs", "assessment"
    ]].copy()
    stage_review_view["source"] = stage_review_view["source"].replace(SOURCE_LABELS)
    stage_review_view["known_pairs"] = stage_review_view["known_pairs"].map(lambda n: f"{n}/12")
    stage_review_view["assessment"] = stage_review_view["assessment"].str.removeprefix("Review: ")
    show_table(stage_review_view.rename(columns={
        "patient_id": "Patient", "n_stage": "N-stage", "source": "Source",
        "known_pairs": "Observed", "assessment": "Review finding",
    }))
show_table(n_stage_overview.rename(columns={
    "patient_id": "Patient", "n_stage": "N-stage", "stage_basis": "N-stage basis", **SOURCE_LABELS,
}), details="All patients: N-stage and involved levels on each side")

Patient,N-stage,Source,Observed,Review finding
MID-6,1,Recorded consensus,12/12,N+ with all checked levels negative
MID-6,1,Without pathology,12/12,N+ with all checked levels negative
MID-6,1,With pathology,12/12,N+ with all checked levels negative
MID-10,2c,With pathology,12/12,N+ with all checked levels negative
MID-10,2c,Pathology,12/12,N+ with all checked levels negative
MID-12,1,Recorded consensus,12/12,N+ with all checked levels negative
MID-12,1,Without pathology,12/12,N+ with all checked levels negative
MID-12,1,With pathology,12/12,N+ with all checked levels negative
MID-13,1,Recorded consensus,12/12,N+ with all checked levels negative
MID-13,1,Without pathology,12/12,N+ with all checked levels negative


Patient,N-stage,N-stage basis,Recorded consensus,Without pathology,With pathology,Pathology
MID-1,2,unconfirmed,"ipsi: II, III; contra: II, III","ipsi: II, III, IV, V; contra: II, III","ipsi: II, III, IV, V; contra: II, III",No observations
MID-2,2b,unconfirmed,"ipsi: II, III, V; contra: none observed","ipsi: II, III, V; contra: none observed","ipsi: II, III, V; contra: none observed",No observations
MID-3,2b,unconfirmed,"ipsi: II, III; contra: III","ipsi: II, III; contra: III","ipsi: II, III; contra: III",No observations
MID-4,1,unconfirmed,No observations,No observations,No observations,No observations
MID-5,2c,unconfirmed,"ipsi: I, II, IV; contra: I, II, IV","ipsi: I, II, IV; contra: I, II, IV","ipsi: I, II, IV; contra: I, II, IV",No observations
MID-6,1,unconfirmed,No positives (12/12 observed),No positives (12/12 observed),No positives (12/12 observed),No observations
MID-7,1,unconfirmed,"ipsi: I, II; contra: none observed","ipsi: I, II; contra: none observed","ipsi: I, II; contra: none observed",No observations
MID-8,1,unconfirmed,ipsi: II; contra: none observed,ipsi: II; contra: none observed,ipsi: II; contra: none observed,No observations
MID-9,1,unconfirmed,ipsi: II; contra: none observed,ipsi: II; contra: none observed,ipsi: II; contra: none observed,No observations
MID-10,2c,unconfirmed,ipsi: II; contra: II,"ipsi: I, II, III; contra: none observed",No positives (12/12 observed),No positives (12/12 observed)


## Workable data

In [22]:
disagreement_report = analysis["disagreements"].copy()
disagreement_report['patient_id'].unique()

<StringArray>
['MID-1', 'MID-10', 'MID-22', 'MID-24', 'MID-26', 'MID-32', 'MID-34']
Length: 7, dtype: str

one pathology issue patient (10) the rest 6 patients have a consensus vs ct problem (ct being the only diagnostics)

In [23]:
data = analysis["raw"]
excluded_ids = disagreement_report["patient_id"].unique()

evaluation_data = data.loc[~data[("patient", "core", "id")].isin(excluded_ids)].copy()

print(f"Retained {len(evaluation_data)} of {len(data)} patients")

Retained 35 of 42 patients


In [24]:
evaluation_data.to_csv(PROJECT_DIR / "ready_tables" / f"{DATASET_NAME}_evaluation_data.csv", index=False)